# 03 — аналіз SSL-претренування ViT-S/16 (MAE + InfoNCE)

Ноутбук лише читає артефакти: `runs/ssl_*/` (`metrics.csv`, `config.yaml`, `recon/*.png`), `reports/tables/probe_<tag>.json` та `runs/probe/<tag>/features_val.pt`, і будує графіки функціями з `malaria_ssl.ssl.viz`. Probe: fit на train-кропах, оцінка на val-кропах; test не використовується.

Як отримати артефакти:
```
python scripts/pretrain.py --config configs/ssl/joint.yaml
python scripts/probe.py --run runs/ssl_joint
python scripts/probe.py --ref random
python scripts/probe.py --ref imagenet
```

In [ ]:
import pandas as pd

from malaria_ssl.ssl import viz

pd.set_option('display.precision', 3)
runs = viz.find_ssl_runs()
probes = viz.load_probes()
print('runs:', [r['name'] for r in runs])
print('probes:', [p['tag'] for p in probes])

## Криві втрат

In [ ]:
viz.plot_losses(runs);

## Ефективні ваги, learning rate, норма градієнта

Для `uncertainty` ваги = exp(−s_i); для `fixed` — λ з конфігу.

In [ ]:
viz.plot_weights_lr(runs);

## Діагностика колапсу (фіксовані val-пари)

`std × √d` ≈ 1 — ембединги розкидані по сфері, → 0 — повний колапс; effective rank ловить розмірнісний колапс.

In [ ]:
viz.plot_collapse(runs, 'rep');

In [ ]:
viz.plot_collapse([r for r in runs if 'proj/erank' in r['history']], 'proj');

## Light probe під час тренування

In [ ]:
viz.plot_light_probe(runs, 'auprc_infected');

## Реконструкції MAE (рядки: оригінал | маска | реконструкція)

In [ ]:
for r in runs:
    if r['recon']:
        viz.show_recon(r, (0, len(r['recon']) // 2, -1))

## Probe: усі енкодери та референси (val-кропи, 100% міток, 95% bootstrap CI по зображеннях)

In [ ]:
viz.probe_table(probes)

## Recall по класах (linear probe)

In [ ]:
viz.per_class_recall_table(probes, 'linear')

## Low-label: 1% / 10% / 100% міток train-кропів

In [ ]:
viz.plot_low_label(probes, 'auprc_infected');
viz.low_label_table(probes, 'auprc_infected')

## PCA ембедингів клітин (val), кольори — класи

In [ ]:
viz.plot_pca([p['tag'] for p in probes], 'val');